<a href="https://colab.research.google.com/github/lizlorenzana931109-art/finpredict-nlp/blob/main/notebooks/01_model_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FinPredict NLP
## 01 - Comparación de modelos preentrenados

Este notebook documenta la evaluación experimental de tres estrategias
basadas en modelos preentrenados disponibles en Hugging Face para la
identificación temprana de señales de riesgo financiero en textos
relacionados con créditos hipotecarios.

### Estrategias evaluadas

1. **Modelo especializado en español**
   - `Recognai/zeroshot_selectra_medium`

2. **Modelo multilingüe**
   - `MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7`

3. **Modelo especializado en inglés con traducción previa**
   - Clasificador: `facebook/bart-large-mnli`
   - Traductor: `Helsinki-NLP/opus-mt-es-en`

### Métricas principales

- Accuracy
- Precision
- Recall
- F1-score
- Latencia de inferencia
- Matriz de confusión

También se analizará el desempeño por:

- dificultad lingüística;
- lenguaje estándar;
- lenguaje local.

La selección final considerará tanto el desempeño técnico como la
eficiencia y viabilidad operativa para FinPredict NLP.

## 1. Configuración del entorno

La experimentación se realiza en Google Colab utilizando GPU para acelerar
la inferencia de los modelos preentrenados.

Primero se verifica la disponibilidad del hardware asignado.

In [1]:
!nvidia-smi

Sun Oct 11 21:00:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch

print("PyTorch:", torch.__version__)
print("GPU disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
GPU disponible: True
GPU: Tesla T4


## 2. Carga del repositorio

Se utiliza el repositorio oficial de FinPredict NLP para garantizar que
el experimento utilice el dataset validado y los archivos versionados
del proyecto.

In [3]:
!git clone https://github.com/lizlorenzana931109-art/finpredict-nlp.git
%cd finpredict-nlp

Cloning into 'finpredict-nlp'...
remote: Enumerating objects: 69, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 69 (delta 26), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (69/69), 80.25 KiB | 702.00 KiB/s, done.
Resolving deltas: 100% (26/26), done.
/content/finpredict-nlp


## 3. Instalación de dependencias

Se instalan las librerías necesarias para la carga de modelos,
autenticación, traducción, inferencia y evaluación.

Se utiliza una versión 4.x de Transformers para mantener compatibilidad
con el pipeline de traducción utilizado en la estrategia inglés +
traducción.

In [4]:
!pip install -q \
    "transformers<5.0.0" \
    accelerate \
    datasets \
    evaluate \
    sentencepiece \
    sacremoses \
    scikit-learn \
    huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 120.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 108.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 4. Autenticación en Hugging Face Hub

La autenticación se realiza de manera interactiva para evitar almacenar
credenciales o tokens dentro del código versionado en GitHub.

In [6]:
from huggingface_hub import notebook_login

notebook_login()

In [7]:
from huggingface_hub import whoami

usuario = whoami()
print("Autenticación correcta:", usuario["name"])

Autenticación correcta: Lizblor


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## 5. Carga del dataset de evaluación

Se carga el dataset previamente generado y validado en FinPredict NLP.

Antes de ejecutar la evaluación completa se utilizará una muestra pequeña
y balanceada para validar el comportamiento del pipeline, las etiquetas
y la configuración de inferencia.

In [8]:
import pandas as pd

df = pd.read_csv("data/finpredict_nlp_dataset.csv")

print("Dimensiones:", df.shape)
display(df.head())

Dimensiones: (2000, 7)


,id,texto,etiqueta,clase,tipo_senal,dificultad,expresion_local
0,FP0001,Quería comentar que hice una transferencia par...,0,SIN_SEÑAL_RIESGO,confirmacion_pago,baja,0
1,FP0002,Necesito informar que quiero saber cuánto sald...,0,SIN_SEÑAL_RIESGO,consulta_informacion,baja,0
2,FP0003,Quería comentar que no cuento con suficiente e...,1,SEÑAL_RIESGO,problemas_liquidez,media,0
3,FP0004,Quería comentar que tuvimos gastos familiares ...,1,SEÑAL_RIESGO,dificultades_familiares,baja,0
4,FP0005,Necesito informar que por ahora no tengo sufic...,1,SEÑAL_RIESGO,problemas_liquidez,alta,0


## 6. Muestra de prueba

Se seleccionan inicialmente 10 registros:

- 5 con `SEÑAL_RIESGO`;
- 5 con `SIN_SEÑAL_RIESGO`.

La muestra se utiliza únicamente para comprobar que el pipeline funciona
correctamente antes de ejecutar la inferencia sobre los 2,000 registros.

In [10]:
muestra_riesgo = (
    df[df["clase"] == "SEÑAL_RIESGO"]
    .sample(n=5, random_state=42)
)

muestra_sin_riesgo = (
    df[df["clase"] == "SIN_SEÑAL_RIESGO"]
    .sample(n=5, random_state=42)
)

muestra_prueba = (
    pd.concat([muestra_riesgo, muestra_sin_riesgo])
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

display(
    muestra_prueba[
        [
            "id",
            "texto",
            "etiqueta",
            "clase",
            "dificultad",
            "expresion_local"
        ]
    ]
)

,id,texto,etiqueta,clase,dificultad,expresion_local
0,FP1287,Nomás para comentar: quisiera comentar que nec...,0,SIN_SEÑAL_RIESGO,alta,1
1,FP1503,Actualmente quisiera solicitar un diferimiento...,1,SEÑAL_RIESGO,media,0
2,FP1005,Quisiera comentar que ya cubrí el monto comple...,0,SIN_SEÑAL_RIESGO,media,0
3,FP1082,"Por el momento me redujeron el sueldo, por lo ...",1,SEÑAL_RIESGO,media,0
4,FP1454,Actualmente quiero entender qué significa una ...,0,SIN_SEÑAL_RIESGO,alta,0
5,FP1508,Quería comentar que una emergencia consumió pa...,1,SEÑAL_RIESGO,media,0
6,FP0797,Quisiera comentar que quisiera descargar mi es...,0,SIN_SEÑAL_RIESGO,media,0
7,FP0850,"Necesito informar que me quedé sin chamba, por...",1,SEÑAL_RIESGO,alta,1
8,FP1348,En este momento quiero revisar si puedo reestr...,1,SEÑAL_RIESGO,baja,0
9,FP1451,Actualmente ¿qué opciones existirían si algún ...,0,SIN_SEÑAL_RIESGO,alta,0


## 7. Configuración de etiquetas zero-shot

Los modelos evaluados utilizarán clasificación zero-shot para asignar
cada texto a una de las dos categorías definidas para FinPredict NLP.

Para mantener una formulación semánticamente natural, se utilizan las
siguientes etiquetas candidatas:

- `riesgo financiero`
- `estabilidad financiera`

Para los modelos que procesan directamente textos en español se utilizará
la plantilla:

`Este texto refleja {}.`

Esta configuración permite evaluar los modelos sin realizar un proceso de
fine-tuning sobre el dataset del proyecto.

In [12]:
ETIQUETAS_CANDIDATAS = [
    "riesgo financiero",
    "estabilidad financiera",
]

PLANTILLA_ES = "Este texto refleja {}."

## 8. Modelo 1 - SELECTRA especializado en español

Se evalúa `Recognai/zeroshot_selectra_medium` como representante de una
estrategia especializada en español.

El modelo está disponible en Hugging Face para tareas de
`zero-shot-classification`, por lo que puede utilizarse sin realizar
fine-tuning sobre el dataset de FinPredict NLP. :chatgpt-content-reference{index="0"}

La inferencia se realiza directamente sobre los textos originales en español,
sin incorporar una etapa de traducción.

In [13]:
  from transformers import pipeline
import torch

MODELO_SELECTRA = "Recognai/zeroshot_selectra_medium"

device = 0 if torch.cuda.is_available() else -1

clasificador_selectra = pipeline(
    task="zero-shot-classification",
    model=MODELO_SELECTRA,
    device=device
)

print("Modelo cargado correctamente.")
print("Dispositivo:", "GPU" if device == 0 else "CPU")

config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/163M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/337 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cuda:0


Modelo cargado correctamente.
Dispositivo: GPU


### 8.1 Prueba inicial con 10 registros

Antes de ejecutar la inferencia sobre los 2,000 registros, se realiza una
prueba controlada con 10 observaciones balanceadas.

El objetivo es verificar:

- funcionamiento del pipeline;
- interpretación de las etiquetas;
- coherencia de las predicciones;
- configuración de la plantilla zero-shot.

In [15]:
resultados_selectra = []

for _, fila in muestra_prueba.iterrows():

    resultado = clasificador_selectra(
        fila["texto"],
        candidate_labels=ETIQUETAS_CANDIDATAS,
        hypothesis_template=PLANTILLA_ES
    )

    prediccion = resultado["labels"][0]
    confianza = resultado["scores"][0]

    resultados_selectra.append({
        "id": fila["id"],
        "texto": fila["texto"],
        "clase_real": fila["clase"],
        "prediccion_modelo": prediccion,
        "confianza": round(confianza, 4)
    })

df_prueba_selectra = pd.DataFrame(resultados_selectra)

display(df_prueba_selectra)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


,id,texto,clase_real,prediccion_modelo,confianza
0,FP1287,Nomás para comentar: quisiera comentar que nec...,SIN_SEÑAL_RIESGO,riesgo financiero,0.9277
1,FP1503,Actualmente quisiera solicitar un diferimiento...,SEÑAL_RIESGO,riesgo financiero,0.7834
2,FP1005,Quisiera comentar que ya cubrí el monto comple...,SIN_SEÑAL_RIESGO,riesgo financiero,0.8884
3,FP1082,"Por el momento me redujeron el sueldo, por lo ...",SEÑAL_RIESGO,riesgo financiero,0.9252
4,FP1454,Actualmente quiero entender qué significa una ...,SIN_SEÑAL_RIESGO,riesgo financiero,0.6387
5,FP1508,Quería comentar que una emergencia consumió pa...,SEÑAL_RIESGO,riesgo financiero,0.9568
6,FP0797,Quisiera comentar que quisiera descargar mi es...,SIN_SEÑAL_RIESGO,estabilidad financiera,0.6027
7,FP0850,"Necesito informar que me quedé sin chamba, por...",SEÑAL_RIESGO,riesgo financiero,0.8677
8,FP1348,En este momento quiero revisar si puedo reestr...,SEÑAL_RIESGO,riesgo financiero,0.8583
9,FP1451,Actualmente ¿qué opciones existirían si algún ...,SIN_SEÑAL_RIESGO,riesgo financiero,0.7404


### 8.2 Calibración semántica de las etiquetas

La prueba inicial mostró una tendencia del modelo a clasificar la mayoría de
los textos como `riesgo financiero`.

Esto sugiere que las etiquetas iniciales son demasiado generales para el
objetivo específico de FinPredict NLP.

Debido a que la variable objetivo representa la presencia o ausencia de una
señal actual de dificultad de pago, se prueba una formulación más cercana a
las reglas de etiquetado del dataset.

In [16]:
ETIQUETAS_SELECTRA_V2 = [
    "dificultad actual de pago",
    "ausencia de dificultad actual de pago",
]

PLANTILLA_SELECTRA_V2 = "Este texto refleja {}."

In [18]:
resultados_selectra_v2 = []

for _, fila in muestra_prueba.iterrows():

    resultado = clasificador_selectra(
        fila["texto"],
        candidate_labels=ETIQUETAS_SELECTRA_V2,
        hypothesis_template=PLANTILLA_SELECTRA_V2
    )

    resultados_selectra_v2.append({
        "id": fila["id"],
        "texto": fila["texto"],
        "clase_real": fila["clase"],
        "prediccion_modelo": resultado["labels"][0],
        "confianza": round(resultado["scores"][0], 4)
    })

df_prueba_selectra_v2 = pd.DataFrame(resultados_selectra_v2)

display(df_prueba_selectra_v2)

,id,texto,clase_real,prediccion_modelo,confianza
0,FP1287,Nomás para comentar: quisiera comentar que nec...,SIN_SEÑAL_RIESGO,dificultad actual de pago,0.9710
1,FP1503,Actualmente quisiera solicitar un diferimiento...,SEÑAL_RIESGO,dificultad actual de pago,0.8983
2,FP1005,Quisiera comentar que ya cubrí el monto comple...,SIN_SEÑAL_RIESGO,ausencia de dificultad actual de pago,0.6616
3,FP1082,"Por el momento me redujeron el sueldo, por lo ...",SEÑAL_RIESGO,dificultad actual de pago,0.9883
4,FP1454,Actualmente quiero entender qué significa una ...,SIN_SEÑAL_RIESGO,dificultad actual de pago,0.9251
5,FP1508,Quería comentar que una emergencia consumió pa...,SEÑAL_RIESGO,dificultad actual de pago,0.9832
6,FP0797,Quisiera comentar que quisiera descargar mi es...,SIN_SEÑAL_RIESGO,dificultad actual de pago,0.7128
7,FP0850,"Necesito informar que me quedé sin chamba, por...",SEÑAL_RIESGO,dificultad actual de pago,0.9826
8,FP1348,En este momento quiero revisar si puedo reestr...,SEÑAL_RIESGO,dificultad actual de pago,0.9345
9,FP1451,Actualmente ¿qué opciones existirían si algún ...,SIN_SEÑAL_RIESGO,dificultad actual de pago,0.8178


### 8.3 Segunda calibración semántica

La segunda formulación mantuvo una tendencia hacia la clase asociada con
dificultad de pago.

Aunque el modelo identificó todos los casos reales con señal de riesgo,
presentó falsos positivos en consultas administrativas, preventivas y
situaciones sin una dificultad financiera vigente.

Para reducir la dependencia de formulaciones negativas, se establece una
tercera configuración en la que ambas etiquetas describen positivamente
el tipo de situación representada en el texto.

In [19]:
ETIQUETAS_SELECTRA_V3 = [
    "dificultad financiera vigente",
    "gestión administrativa, preventiva o situación ya resuelta",
]

PLANTILLA_SELECTRA_V3 = "El mensaje describe {}."

In [21]:
resultados_selectra_v3 = []

for _, fila in muestra_prueba.iterrows():

    resultado = clasificador_selectra(
        fila["texto"],
        candidate_labels=ETIQUETAS_SELECTRA_V3,
        hypothesis_template=PLANTILLA_SELECTRA_V3
    )

    resultados_selectra_v3.append({
        "id": fila["id"],
        "texto": fila["texto"],
        "clase_real": fila["clase"],
        "prediccion_modelo": resultado["labels"][0],
        "confianza": round(resultado["scores"][0], 4)
    })

df_prueba_selectra_v3 = pd.DataFrame(resultados_selectra_v3)

display(df_prueba_selectra_v3)

,id,texto,clase_real,prediccion_modelo,confianza
0,FP1287,Nomás para comentar: quisiera comentar que nec...,SIN_SEÑAL_RIESGO,dificultad financiera vigente,0.9875
1,FP1503,Actualmente quisiera solicitar un diferimiento...,SEÑAL_RIESGO,dificultad financiera vigente,0.9515
2,FP1005,Quisiera comentar que ya cubrí el monto comple...,SIN_SEÑAL_RIESGO,"gestión administrativa, preventiva o situación...",0.5011
3,FP1082,"Por el momento me redujeron el sueldo, por lo ...",SEÑAL_RIESGO,dificultad financiera vigente,0.9834
4,FP1454,Actualmente quiero entender qué significa una ...,SIN_SEÑAL_RIESGO,"gestión administrativa, preventiva o situación...",0.7867
5,FP1508,Quería comentar que una emergencia consumió pa...,SEÑAL_RIESGO,dificultad financiera vigente,0.9880
6,FP0797,Quisiera comentar que quisiera descargar mi es...,SIN_SEÑAL_RIESGO,"gestión administrativa, preventiva o situación...",0.9141
7,FP0850,"Necesito informar que me quedé sin chamba, por...",SEÑAL_RIESGO,dificultad financiera vigente,0.9535
8,FP1348,En este momento quiero revisar si puedo reestr...,SEÑAL_RIESGO,dificultad financiera vigente,0.8253
9,FP1451,Actualmente ¿qué opciones existirían si algún ...,SIN_SEÑAL_RIESGO,dificultad financiera vigente,0.8679


### 8.4 Configuración seleccionada para SELECTRA

La tercera formulación presentó una mejora respecto a las configuraciones
anteriores y redujo la tendencia del modelo a clasificar consultas
administrativas como señales de riesgo.

La configuración seleccionada para la evaluación final es:

- `dificultad financiera vigente`
- `gestión administrativa, preventiva o situación ya resuelta`

Plantilla:

`El mensaje describe {}.`

En la muestra preliminar de 10 registros, esta configuración clasificó
correctamente 8 observaciones.

Los dos errores restantes correspondieron a textos sin señal de riesgo que
incluían vocabulario relacionado con dificultad financiera, lo cual sugiere
una posible sensibilidad del modelo a determinados términos aun cuando el
contexto sea preventivo o administrativo.

Para evitar ajustar adicionalmente la configuración a esta misma muestra,
las etiquetas quedan fijadas a partir de este punto.

In [22]:
# IDs utilizados durante la calibración semántica
IDS_CALIBRACION = muestra_prueba["id"].tolist()

# Dataset utilizado para la evaluación final
df_evaluacion = df[
    ~df["id"].isin(IDS_CALIBRACION)
].copy()

print("Registros de calibración:", len(IDS_CALIBRACION))
print("Registros para evaluación final:", len(df_evaluacion))

Registros de calibración: 10
Registros para evaluación final: 1990


### 8.5 Evaluación completa de SELECTRA

Después de fijar la configuración zero-shot, el modelo se evalúa sobre los
1,990 registros que no participaron en la calibración semántica.

Para garantizar una comparación posterior consistente entre modelos:

- se utiliza el mismo conjunto de evaluación;
- se mantiene fija la configuración de etiquetas;
- se registra el tiempo total de inferencia;
- se calcula la latencia promedio por registro;
- se conservan las predicciones para calcular las métricas de desempeño.

Los registros utilizados durante la calibración quedan excluidos de las
métricas finales.

In [23]:
import time
import torch

# Configuración definitiva de SELECTRA
ETIQUETAS_SELECTRA_FINAL = [
    "dificultad financiera vigente",
    "gestión administrativa, preventiva o situación ya resuelta",
]

PLANTILLA_SELECTRA_FINAL = "El mensaje describe {}."

# Textos de evaluación
textos_evaluacion = df_evaluacion["texto"].tolist()

print("Registros a evaluar:", len(textos_evaluacion))

Registros a evaluar: 1990


In [24]:
# Warm-up del modelo

_ = clasificador_selectra(
    textos_evaluacion[0],
    candidate_labels=ETIQUETAS_SELECTRA_FINAL,
    hypothesis_template=PLANTILLA_SELECTRA_FINAL
)

if torch.cuda.is_available():
    torch.cuda.synchronize()

print("✅ Warm-up completado.")

✅ Warm-up completado.


In [25]:
# ============================================================
# INFERENCIA COMPLETA - SELECTRA
# ============================================================

if torch.cuda.is_available():
    torch.cuda.synchronize()

inicio = time.perf_counter()

resultados = clasificador_selectra(
    textos_evaluacion,
    candidate_labels=ETIQUETAS_SELECTRA_FINAL,
    hypothesis_template=PLANTILLA_SELECTRA_FINAL,
    batch_size=16
)

if torch.cuda.is_available():
    torch.cuda.synchronize()

fin = time.perf_counter()

tiempo_total_selectra = fin - inicio
latencia_media_selectra = (
    tiempo_total_selectra / len(textos_evaluacion)
) * 1000

throughput_selectra = (
    len(textos_evaluacion) / tiempo_total_selectra
)

print(f"Tiempo total: {tiempo_total_selectra:.2f} segundos")
print(f"Latencia promedio: {latencia_media_selectra:.2f} ms/registro")
print(f"Throughput: {throughput_selectra:.2f} registros/segundo")

Tiempo total: 5.16 segundos
Latencia promedio: 2.59 ms/registro
Throughput: 385.91 registros/segundo


In [26]:
# Extraer etiqueta y score principal de cada predicción

predicciones_selectra = [
    resultado["labels"][0]
    for resultado in resultados
]

scores_selectra = [
    resultado["scores"][0]
    for resultado in resultados
]

MAPEO_SELECTRA = {
    "dificultad financiera vigente": 1,
    "gestión administrativa, preventiva o situación ya resuelta": 0,
}

df_evaluacion["pred_selectra"] = [
    MAPEO_SELECTRA[pred]
    for pred in predicciones_selectra
]

df_evaluacion["score_selectra"] = scores_selectra

display(
    df_evaluacion[
        [
            "id",
            "texto",
            "etiqueta",
            "pred_selectra",
            "score_selectra",
        ]
    ].head(10)
)

,id,texto,etiqueta,pred_selectra,score_selectra
0,FP0001,Quería comentar que hice una transferencia par...,0,1,0.678197
1,FP0002,Necesito informar que quiero saber cuánto sald...,0,1,0.930607
2,FP0003,Quería comentar que no cuento con suficiente e...,1,1,0.989870
3,FP0004,Quería comentar que tuvimos gastos familiares ...,1,1,0.989207
4,FP0005,Necesito informar que por ahora no tengo sufic...,1,1,0.970020
5,FP0006,Actualmente me está costando cubrir la mensual...,1,1,0.986817
6,FP0007,Necesito informar que necesito revisar el sald...,0,1,0.969902
7,FP0008,Actualmente necesito posponer temporalmente el...,1,1,0.991125
8,FP0009,Por el momento quiero consultar el detalle de ...,0,1,0.530058
9,FP0010,Quería checar algo: por el momento quisiera ac...,0,0,0.680088


### 8.6 Métricas de desempeño

Se calculan las métricas requeridas para evaluar el desempeño del modelo.

Para `precision`, `recall` y `F1-score`, la clase positiva corresponde a:

`SEÑAL_RIESGO = 1`

Esto permite dar especial atención a la capacidad del modelo para identificar
textos que contienen señales potenciales de deterioro financiero.

In [27]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

y_true = df_evaluacion["etiqueta"]
y_pred = df_evaluacion["pred_selectra"]

accuracy_selectra = accuracy_score(y_true, y_pred)
precision_selectra = precision_score(y_true, y_pred)
recall_selectra = recall_score(y_true, y_pred)
f1_selectra = f1_score(y_true, y_pred)

print("=== SELECTRA - RESULTADOS ===")
print(f"Accuracy : {accuracy_selectra:.4f}")
print(f"Precision: {precision_selectra:.4f}")
print(f"Recall   : {recall_selectra:.4f}")
print(f"F1-score : {f1_selectra:.4f}")

print("\nMatriz de confusión:")
print(confusion_matrix(y_true, y_pred))

print("\nReporte de clasificación:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "SIN_SEÑAL_RIESGO",
            "SEÑAL_RIESGO"
        ]
    )
)

=== SELECTRA - RESULTADOS ===
Accuracy : 0.6141
Precision: 0.5653
Recall   : 0.9879
F1-score : 0.7191

Matriz de confusión:
[[239 756]
 [ 12 983]]

Reporte de clasificación:
                  precision    recall  f1-score   support

SIN_SEÑAL_RIESGO       0.95      0.24      0.38       995
    SEÑAL_RIESGO       0.57      0.99      0.72       995

        accuracy                           0.61      1990
       macro avg       0.76      0.61      0.55      1990
    weighted avg       0.76      0.61      0.55      1990



### 8.7 Interpretación parcial de SELECTRA

SELECTRA presentó una alta sensibilidad para identificar textos con señal
de riesgo financiero.

El modelo alcanzó un recall de 98.79 % para la clase `SEÑAL_RIESGO`,
identificando correctamente 983 de los 995 casos positivos del conjunto
de evaluación.

Sin embargo, presentó una tendencia significativa a clasificar textos como
riesgo, generando 756 falsos positivos y reconociendo correctamente únicamente
239 de los 995 registros sin señal de riesgo.

Este comportamiento produjo:

- Accuracy: 61.41 %
- Precision de riesgo: 56.53 %
- Recall de riesgo: 98.79 %
- F1-score de riesgo: 71.91 %

Desde una perspectiva de negocio, el modelo ofrece una alta capacidad para
evitar falsos negativos, característica relevante en un sistema de detección
temprana. No obstante, la elevada cantidad de falsos positivos podría generar
una carga operativa considerable si cada señal requiere revisión o intervención
humana.

Por este motivo, SELECTRA deberá compararse con los modelos restantes antes
de determinar si su elevada sensibilidad compensa su baja especificidad.

In [28]:
resultados_modelos = []

resultados_modelos.append({
    "modelo": "Recognai/zeroshot_selectra_medium",
    "estrategia": "Español",
    "accuracy": accuracy_selectra,
    "precision": precision_selectra,
    "recall": recall_selectra,
    "f1": f1_selectra,
    "tiempo_total_s": tiempo_total_selectra,
    "latencia_ms": latencia_media_selectra,
    "throughput_reg_s": throughput_selectra
})

pd.DataFrame(resultados_modelos)

,modelo,estrategia,accuracy,precision,recall,f1,tiempo_total_s,latencia_ms,throughput_reg_s
0,Recognai/zeroshot_selectra_medium,Español,0.61407,0.565267,0.98794,0.719093,5.156695,2.591304,385.906089


## 9. Modelo 2 - mDeBERTa multilingüe

Se evalúa el modelo
`MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7`
como representante de una estrategia multilingüe.

El modelo procesa directamente los textos originales en español, por lo que
no requiere una etapa previa de traducción.

Para mantener comparabilidad con SELECTRA se utiliza:

- el mismo conjunto de evaluación de 1,990 registros;
- las mismas etiquetas semánticas;
- la misma plantilla zero-shot;
- las mismas métricas;
- la misma estrategia de medición de latencia.

La configuración de etiquetas se mantiene fija para evitar ajustar el
experimento individualmente a cada modelo.

In [29]:
# Liberar memoria utilizada por SELECTRA

del clasificador_selectra

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("✅ Memoria de SELECTRA liberada.")

✅ Memoria de SELECTRA liberada.


In [30]:
from transformers import pipeline

MODELO_MDEBERTA = (
    "MoritzLaurer/"
    "mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
)

clasificador_mdeberta = pipeline(
    task="zero-shot-classification",
    model=MODELO_MDEBERTA,
    device=0 if torch.cuda.is_available() else -1
)

print("✅ mDeBERTa cargado correctamente.")
print(
    "Dispositivo:",
    "GPU" if torch.cuda.is_available() else "CPU"
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.3M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Device set to use cuda:0


✅ mDeBERTa cargado correctamente.
Dispositivo: GPU


### 9.1 Prueba técnica sobre la muestra de calibración

La muestra utilizada previamente se emplea únicamente para comprobar el
funcionamiento técnico del pipeline.

Las etiquetas no se modificarán con base en estos resultados.

In [31]:
resultados_mdeberta_prueba = []

for _, fila in muestra_prueba.iterrows():

    resultado = clasificador_mdeberta(
        fila["texto"],
        candidate_labels=ETIQUETAS_SELECTRA_FINAL,
        hypothesis_template=PLANTILLA_SELECTRA_FINAL
    )

    resultados_mdeberta_prueba.append({
        "id": fila["id"],
        "clase_real": fila["clase"],
        "prediccion_modelo": resultado["labels"][0],
        "score_modelo": round(resultado["scores"][0], 4)
    })

df_prueba_mdeberta = pd.DataFrame(
    resultados_mdeberta_prueba
)

display(df_prueba_mdeberta)

,id,clase_real,prediccion_modelo,score_modelo
0,FP1287,SIN_SEÑAL_RIESGO,dificultad financiera vigente,0.9942
1,FP1503,SEÑAL_RIESGO,dificultad financiera vigente,0.9996
2,FP1005,SIN_SEÑAL_RIESGO,"gestión administrativa, preventiva o situación...",0.7256
3,FP1082,SEÑAL_RIESGO,dificultad financiera vigente,0.9997
4,FP1454,SIN_SEÑAL_RIESGO,dificultad financiera vigente,0.8521
5,FP1508,SEÑAL_RIESGO,dificultad financiera vigente,0.9994
6,FP0797,SIN_SEÑAL_RIESGO,dificultad financiera vigente,0.7434
7,FP0850,SEÑAL_RIESGO,dificultad financiera vigente,0.9998
8,FP1348,SEÑAL_RIESGO,dificultad financiera vigente,0.9902
9,FP1451,SIN_SEÑAL_RIESGO,dificultad financiera vigente,0.9972


### 9.2 Evaluación completa de mDeBERTa

In [32]:
_ = clasificador_mdeberta(
    textos_evaluacion[0],
    candidate_labels=ETIQUETAS_SELECTRA_FINAL,
    hypothesis_template=PLANTILLA_SELECTRA_FINAL
)

if torch.cuda.is_available():
    torch.cuda.synchronize()

print("✅ Warm-up completado.")

✅ Warm-up completado.


In [33]:
if torch.cuda.is_available():
    torch.cuda.synchronize()

inicio = time.perf_counter()

resultados_mdeberta = clasificador_mdeberta(
    textos_evaluacion,
    candidate_labels=ETIQUETAS_SELECTRA_FINAL,
    hypothesis_template=PLANTILLA_SELECTRA_FINAL,
    batch_size=16
)

if torch.cuda.is_available():
    torch.cuda.synchronize()

fin = time.perf_counter()

tiempo_total_mdeberta = fin - inicio

latencia_media_mdeberta = (
    tiempo_total_mdeberta / len(textos_evaluacion)
) * 1000

throughput_mdeberta = (
    len(textos_evaluacion) / tiempo_total_mdeberta
)

print(f"Tiempo total: {tiempo_total_mdeberta:.2f} segundos")
print(f"Latencia promedio: {latencia_media_mdeberta:.2f} ms/registro")
print(f"Throughput: {throughput_mdeberta:.2f} registros/segundo")

Tiempo total: 15.00 segundos
Latencia promedio: 7.54 ms/registro
Throughput: 132.68 registros/segundo


In [34]:
predicciones_mdeberta = [
    resultado["labels"][0]
    for resultado in resultados_mdeberta
]

scores_mdeberta = [
    resultado["scores"][0]
    for resultado in resultados_mdeberta
]

MAPEO_MDEBERTA = {
    "dificultad financiera vigente": 1,
    "gestión administrativa, preventiva o situación ya resuelta": 0,
}

df_evaluacion["pred_mdeberta"] = [
    MAPEO_MDEBERTA[pred]
    for pred in predicciones_mdeberta
]

df_evaluacion["score_mdeberta"] = scores_mdeberta

In [35]:
y_true = df_evaluacion["etiqueta"]
y_pred_mdeberta = df_evaluacion["pred_mdeberta"]

accuracy_mdeberta = accuracy_score(
    y_true,
    y_pred_mdeberta
)

precision_mdeberta = precision_score(
    y_true,
    y_pred_mdeberta
)

recall_mdeberta = recall_score(
    y_true,
    y_pred_mdeberta
)

f1_mdeberta = f1_score(
    y_true,
    y_pred_mdeberta
)

print("=== mDeBERTa - RESULTADOS ===")
print(f"Accuracy : {accuracy_mdeberta:.4f}")
print(f"Precision: {precision_mdeberta:.4f}")
print(f"Recall   : {recall_mdeberta:.4f}")
print(f"F1-score : {f1_mdeberta:.4f}")

print("\nMatriz de confusión:")
print(
    confusion_matrix(
        y_true,
        y_pred_mdeberta
    )
)

=== mDeBERTa - RESULTADOS ===
Accuracy : 0.5859
Precision: 0.5470
Recall   : 1.0000
F1-score : 0.7072

Matriz de confusión:
[[171 824]
 [  0 995]]


In [36]:
resultados_modelos.append({
    "modelo": MODELO_MDEBERTA,
    "estrategia": "Multilingüe",
    "accuracy": accuracy_mdeberta,
    "precision": precision_mdeberta,
    "recall": recall_mdeberta,
    "f1": f1_mdeberta,
    "tiempo_total_s": tiempo_total_mdeberta,
    "latencia_ms": latencia_media_mdeberta,
    "throughput_reg_s": throughput_mdeberta
})

display(
    pd.DataFrame(resultados_modelos)
)

,modelo,estrategia,accuracy,precision,recall,f1,tiempo_total_s,latencia_ms,throughput_reg_s
0,Recognai/zeroshot_selectra_medium,Español,0.61407,0.565267,0.98794,0.719093,5.156695,2.591304,385.906089
1,MoritzLaurer/mDeBERTa-v3-base-xnli-multilingua...,Multilingüe,0.58593,0.547004,1.00000,0.707178,14.998869,7.537120,132.676667


## 10. Revisión del protocolo experimental

Las primeras ejecuciones permitieron comprobar el funcionamiento técnico de
SELECTRA y mDeBERTa. Sin embargo, la formulación semántica utilizada para la
clasificación zero-shot fue ajustada inicialmente observando el comportamiento
de SELECTRA.

Para garantizar una comparación equitativa entre los tres modelos, se redefine
el protocolo experimental en tres grupos independientes:

1. **Muestra piloto:** 10 registros utilizados para explorar las formulaciones
   iniciales V1, V2 y V3.

2. **Muestra de calibración:** 60 registros adicionales utilizados para
   seleccionar una única formulación zero-shot común para los tres modelos.

3. **Conjunto de evaluación final:** registros restantes, utilizados
   exclusivamente para calcular las métricas finales.

La configuración seleccionada durante la calibración se mantendrá fija durante
la evaluación final.

In [37]:
# ============================================================
# CONSTRUCCIÓN DE LA MUESTRA DE CALIBRACIÓN FORMAL
# ============================================================

# Los 10 registros anteriores quedan registrados como muestra piloto
IDS_PILOTO = muestra_prueba["id"].tolist()

# Excluimos la muestra piloto antes de seleccionar la calibración
df_disponible_calibracion = df[
    ~df["id"].isin(IDS_PILOTO)
].copy()


def seleccionar_calibracion(clase):
    """
    Selecciona 30 registros por clase:

    - 8 dificultad baja, lenguaje estándar
    - 8 dificultad media, lenguaje estándar
    - 8 dificultad alta, lenguaje estándar
    - 6 dificultad alta, lenguaje local

    Total por clase = 30
    """

    baja = df_disponible_calibracion[
        (df_disponible_calibracion["clase"] == clase)
        & (df_disponible_calibracion["dificultad"] == "baja")
        & (df_disponible_calibracion["expresion_local"] == 0)
    ].sample(n=8, random_state=42)

    media = df_disponible_calibracion[
        (df_disponible_calibracion["clase"] == clase)
        & (df_disponible_calibracion["dificultad"] == "media")
        & (df_disponible_calibracion["expresion_local"] == 0)
    ].sample(n=8, random_state=42)

    alta_estandar = df_disponible_calibracion[
        (df_disponible_calibracion["clase"] == clase)
        & (df_disponible_calibracion["dificultad"] == "alta")
        & (df_disponible_calibracion["expresion_local"] == 0)
    ].sample(n=8, random_state=42)

    alta_local = df_disponible_calibracion[
        (df_disponible_calibracion["clase"] == clase)
        & (df_disponible_calibracion["dificultad"] == "alta")
        & (df_disponible_calibracion["expresion_local"] == 1)
    ].sample(n=6, random_state=42)

    return pd.concat([
        baja,
        media,
        alta_estandar,
        alta_local
    ])


calibracion_riesgo = seleccionar_calibracion(
    "SEÑAL_RIESGO"
)

calibracion_sin_riesgo = seleccionar_calibracion(
    "SIN_SEÑAL_RIESGO"
)

df_calibracion = (
    pd.concat([
        calibracion_riesgo,
        calibracion_sin_riesgo
    ])
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

IDS_CALIBRACION_FORMAL = df_calibracion["id"].tolist()

print("Registros de calibración:", len(df_calibracion))

display(
    pd.crosstab(
        df_calibracion["clase"],
        df_calibracion["expresion_local"]
    )
)

display(
    pd.crosstab(
        df_calibracion["clase"],
        df_calibracion["dificultad"]
    )
)

Registros de calibración: 60


expresion_local,0,1
clase,,
SEÑAL_RIESGO,24,6
SIN_SEÑAL_RIESGO,24,6


dificultad,alta,baja,media
clase,,,
SEÑAL_RIESGO,14,8,8
SIN_SEÑAL_RIESGO,14,8,8


In [38]:
# ============================================================
# CONJUNTO DE EVALUACIÓN FINAL
# ============================================================

ids_excluidos = (
    IDS_PILOTO
    + IDS_CALIBRACION_FORMAL
)

df_evaluacion_final = df[
    ~df["id"].isin(ids_excluidos)
].copy()

print("Dataset original:", len(df))
print("Muestra piloto:", len(IDS_PILOTO))
print("Calibración formal:", len(IDS_CALIBRACION_FORMAL))
print("Evaluación final:", len(df_evaluacion_final))

Dataset original: 2000
Muestra piloto: 10
Calibración formal: 60
Evaluación final: 1930


## 11. Calibración común de la configuración zero-shot

Se evalúan tres formulaciones semánticas sobre un conjunto independiente
de 60 registros.

El objetivo es seleccionar posteriormente una única configuración común
para los tres modelos y evitar favorecer a una arquitectura específica.

La calibración se evalúa mediante:

- accuracy;
- F1-score macro;
- precision de la clase de riesgo;
- recall de la clase de riesgo.

Los 60 registros utilizados en esta etapa no formarán parte de la
evaluación final.

In [39]:
CONFIGURACIONES_ZERO_SHOT = {
    "V1": {
        "labels": [
            "riesgo financiero",
            "estabilidad financiera",
        ],
        "template": "Este texto refleja {}.",
        "label_positiva": "riesgo financiero",
    },

    "V2": {
        "labels": [
            "dificultad actual de pago",
            "ausencia de dificultad actual de pago",
        ],
        "template": "Este texto refleja {}.",
        "label_positiva": "dificultad actual de pago",
    },

    "V3": {
        "labels": [
            "dificultad financiera vigente",
            "gestión administrativa, preventiva o situación ya resuelta",
        ],
        "template": "El mensaje describe {}.",
        "label_positiva": "dificultad financiera vigente",
    },
}

In [40]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

def evaluar_configuraciones(
    clasificador,
    nombre_modelo,
    df_calibracion,
):
    """
    Evalúa V1, V2 y V3 sobre la misma muestra de calibración.
    """

    resultados = []

    textos = df_calibracion["texto"].tolist()
    y_true = df_calibracion["etiqueta"].tolist()

    for version, config in CONFIGURACIONES_ZERO_SHOT.items():

        predicciones_raw = clasificador(
            textos,
            candidate_labels=config["labels"],
            hypothesis_template=config["template"],
            batch_size=16,
        )

        y_pred = [
            1 if resultado["labels"][0] == config["label_positiva"]
            else 0
            for resultado in predicciones_raw
        ]

        resultados.append({
            "modelo": nombre_modelo,
            "configuracion": version,
            "accuracy": accuracy_score(y_true, y_pred),
            "precision_riesgo": precision_score(
                y_true,
                y_pred,
                zero_division=0,
            ),
            "recall_riesgo": recall_score(
                y_true,
                y_pred,
                zero_division=0,
            ),
            "f1_riesgo": f1_score(
                y_true,
                y_pred,
                zero_division=0,
            ),
            "f1_macro": f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            ),
        })

    return pd.DataFrame(resultados)

In [41]:
calibracion_mdeberta = evaluar_configuraciones(
    clasificador=clasificador_mdeberta,
    nombre_modelo="mDeBERTa multilingüe",
    df_calibracion=df_calibracion,
)

display(calibracion_mdeberta)

,modelo,configuracion,accuracy,precision_riesgo,recall_riesgo,f1_riesgo,f1_macro
0,mDeBERTa multilingüe,V1,0.650000,0.588235,1.0,0.740741,0.601140
1,mDeBERTa multilingüe,V2,0.750000,0.666667,1.0,0.800000,0.733333
2,mDeBERTa multilingüe,V3,0.616667,0.566038,1.0,0.722892,0.550635


In [42]:
resultados_calibracion = []

resultados_calibracion.append(calibracion_mdeberta)

print("✅ Calibración de mDeBERTa almacenada.")

✅ Calibración de mDeBERTa almacenada.


### 11.1 Calibración de SELECTRA

Se aplica el mismo protocolo de calibración al modelo especializado en
español utilizando exactamente los mismos 60 registros y las mismas tres
configuraciones semánticas.

Esto permite comparar el efecto de la formulación zero-shot sin modificar
el conjunto de calibración.

In [43]:
# Liberar mDeBERTa de la GPU
del clasificador_mdeberta

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("✅ Memoria de mDeBERTa liberada.")

✅ Memoria de mDeBERTa liberada.


In [44]:
  MODELO_SELECTRA = "Recognai/zeroshot_selectra_medium"

clasificador_selectra = pipeline(
    task="zero-shot-classification",
    model=MODELO_SELECTRA,
    device=0 if torch.cuda.is_available() else -1
)

print("✅ SELECTRA cargado correctamente.")

Device set to use cuda:0


✅ SELECTRA cargado correctamente.


In [45]:
calibracion_selectra = evaluar_configuraciones(
    clasificador=clasificador_selectra,
    nombre_modelo="SELECTRA español",
    df_calibracion=df_calibracion,
)

display(calibracion_selectra)

,modelo,configuracion,accuracy,precision_riesgo,recall_riesgo,f1_riesgo,f1_macro
0,SELECTRA español,V1,0.516667,0.509091,0.933333,0.658824,0.415126
1,SELECTRA español,V2,0.650000,0.588235,1.000000,0.740741,0.601140
2,SELECTRA español,V3,0.550000,0.527273,0.966667,0.682353,0.455462


In [46]:
resultados_calibracion.append(calibracion_selectra)

In [47]:
comparacion_calibracion_parcial = pd.concat(
    resultados_calibracion,
    ignore_index=True
)

display(comparacion_calibracion_parcial)

,modelo,configuracion,accuracy,precision_riesgo,recall_riesgo,f1_riesgo,f1_macro
0,mDeBERTa multilingüe,V1,0.650000,0.588235,1.000000,0.740741,0.601140
1,mDeBERTa multilingüe,V2,0.750000,0.666667,1.000000,0.800000,0.733333
2,mDeBERTa multilingüe,V3,0.616667,0.566038,1.000000,0.722892,0.550635
3,SELECTRA español,V1,0.516667,0.509091,0.933333,0.658824,0.415126
4,SELECTRA español,V2,0.650000,0.588235,1.000000,0.740741,0.601140
5,SELECTRA español,V3,0.550000,0.527273,0.966667,0.682353,0.455462


### 11.2 Calibración de BART con traducción español → inglés

La tercera estrategia incorpora una etapa previa de traducción.

Los textos originales en español se traducen al inglés y posteriormente
se procesan mediante `facebook/bart-large-mnli`.

Para mantener equivalencia semántica con las configuraciones evaluadas en
español, V1, V2 y V3 se expresan en inglés conservando el mismo significado.

In [48]:
# Liberar SELECTRA de la GPU
del clasificador_selectra

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("✅ Memoria de SELECTRA liberada.")

✅ Memoria de SELECTRA liberada.


In [49]:
from transformers import pipeline

MODELO_TRADUCCION = "Helsinki-NLP/opus-mt-es-en"
MODELO_BART = "facebook/bart-large-mnli"

traductor_es_en = pipeline(
    task="translation",
    model=MODELO_TRADUCCION,
    device=0 if torch.cuda.is_available() else -1
)

clasificador_bart = pipeline(
    task="zero-shot-classification",
    model=MODELO_BART,
    device=0 if torch.cuda.is_available() else -1
)

print("✅ Traductor español → inglés cargado.")
print("✅ BART-MNLI cargado.")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


✅ Traductor español → inglés cargado.
✅ BART-MNLI cargado.


In [51]:
textos_calibracion_es = df_calibracion["texto"].tolist()

traducciones = traductor_es_en(
    textos_calibracion_es,
    batch_size=16,
    max_length=256
)

textos_calibracion_en = [
    resultado["translation_text"]
    for resultado in traducciones
]

df_calibracion_bart = df_calibracion.copy()
df_calibracion_bart["texto_en"] = textos_calibracion_en

display(
    df_calibracion_bart[
        ["texto", "texto_en", "clase"]
    ].head(10)
)

,texto,texto_en,clase
0,Necesito informar que necesito algunos días ad...,I need to report that I need a few extra days ...,SEÑAL_RIESGO
1,Necesito informar que surgió una emergencia fa...,I need to report that a family emergency arose...,SEÑAL_RIESGO
2,Por el momento necesito una copia de mi estado...,At the moment I need a copy of my statement an...,SIN_SEÑAL_RIESGO
3,Actualmente quisiera actualizar mis datos de c...,Currently I would like to update my contact de...,SIN_SEÑAL_RIESGO
4,Estoy teniendo dificultades para realizar el p...,"I am having difficulty making the payment, bec...",SEÑAL_RIESGO
5,Nomás para comentar: en este momento quisiera ...,Just to comment: right now I would like to kno...,SIN_SEÑAL_RIESGO
6,"Realicé el pago de mi mensualidad. Además, qui...",I made my monthly payment. I also want to make...,SIN_SEÑAL_RIESGO
7,Me atrasé unos días pero el crédito ya está al...,I was late for a few days but the credit is al...,SIN_SEÑAL_RIESGO
8,"No cuento con suficiente efectivo disponible, ...","I don't have enough cash available, because of...",SEÑAL_RIESGO
9,Quería checar algo: quería comentar que me atr...,I wanted to check something: I wanted to comme...,SIN_SEÑAL_RIESGO


In [52]:
CONFIGURACIONES_ZERO_SHOT_EN = {
    "V1": {
        "labels": [
            "financial risk",
            "financial stability",
        ],
        "template": "This text reflects {}.",
        "label_positiva": "financial risk",
    },

    "V2": {
        "labels": [
            "current payment difficulty",
            "absence of current payment difficulty",
        ],
        "template": "This text reflects {}.",
        "label_positiva": "current payment difficulty",
    },

    "V3": {
        "labels": [
            "current financial difficulty",
            "administrative, preventive, or already resolved situation",
        ],
        "template": "The message describes {}.",
        "label_positiva": "current financial difficulty",
    },
}

In [53]:
def evaluar_configuraciones_bart(
    clasificador,
    df_calibracion,
):
    resultados = []

    textos = df_calibracion["texto_en"].tolist()
    y_true = df_calibracion["etiqueta"].tolist()

    for version, config in CONFIGURACIONES_ZERO_SHOT_EN.items():

        predicciones_raw = clasificador(
            textos,
            candidate_labels=config["labels"],
            hypothesis_template=config["template"],
            batch_size=16,
        )

        y_pred = [
            1 if resultado["labels"][0] == config["label_positiva"]
            else 0
            for resultado in predicciones_raw
        ]

        resultados.append({
            "modelo": "BART + traducción",
            "configuracion": version,
            "accuracy": accuracy_score(y_true, y_pred),
            "precision_riesgo": precision_score(
                y_true, y_pred, zero_division=0
            ),
            "recall_riesgo": recall_score(
                y_true, y_pred, zero_division=0
            ),
            "f1_riesgo": f1_score(
                y_true, y_pred, zero_division=0
            ),
            "f1_macro": f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            ),
        })

    return pd.DataFrame(resultados)

In [54]:
calibracion_bart = evaluar_configuraciones_bart(
    clasificador=clasificador_bart,
    df_calibracion=df_calibracion_bart,
)

display(calibracion_bart)

,modelo,configuracion,accuracy,precision_riesgo,recall_riesgo,f1_riesgo,f1_macro
0,BART + traducción,V1,0.583333,0.545455,1.0,0.705882,0.495798
1,BART + traducción,V2,0.750000,0.666667,1.0,0.800000,0.733333
2,BART + traducción,V3,0.850000,0.769231,1.0,0.869565,0.846547


In [55]:
resultados_calibracion.append(calibracion_bart)

comparacion_calibracion = pd.concat(
    resultados_calibracion,
    ignore_index=True
)

display(comparacion_calibracion)

,modelo,configuracion,accuracy,precision_riesgo,recall_riesgo,f1_riesgo,f1_macro
0,mDeBERTa multilingüe,V1,0.650000,0.588235,1.000000,0.740741,0.601140
1,mDeBERTa multilingüe,V2,0.750000,0.666667,1.000000,0.800000,0.733333
2,mDeBERTa multilingüe,V3,0.616667,0.566038,1.000000,0.722892,0.550635
3,SELECTRA español,V1,0.516667,0.509091,0.933333,0.658824,0.415126
4,SELECTRA español,V2,0.650000,0.588235,1.000000,0.740741,0.601140
5,SELECTRA español,V3,0.550000,0.527273,0.966667,0.682353,0.455462
6,BART + traducción,V1,0.583333,0.545455,1.000000,0.705882,0.495798
7,BART + traducción,V2,0.750000,0.666667,1.000000,0.800000,0.733333
8,BART + traducción,V3,0.850000,0.769231,1.000000,0.869565,0.846547


### 11.3 Selección de la configuración común

Los resultados de calibración muestran diferencias en la sensibilidad de
cada arquitectura ante la formulación semántica de la tarea zero-shot.

Aunque BART obtuvo su mejor resultado individual con V3, la configuración
V2 presentó el mejor desempeño agregado entre los tres modelos.

Resultados promedio de V2:

- Accuracy: 71.67 %
- Precision de riesgo: 64.05 %
- Recall de riesgo: 100 %
- F1-score de riesgo: 78.02 %
- F1-score macro: 68.93 %

Además, V2 fue la configuración con mejor desempeño individual para
SELECTRA y mDeBERTa.

Por este motivo, se selecciona V2 como configuración común para la
evaluación final.

La decisión prioriza una comparación equitativa entre arquitecturas y evita
seleccionar una formulación optimizada específicamente para un solo modelo.

A partir de este punto, la configuración queda congelada y no será modificada
con base en los resultados del conjunto de evaluación final.

In [56]:
CONFIGURACION_FINAL_ES = {
    "labels": [
        "dificultad actual de pago",
        "ausencia de dificultad actual de pago",
    ],
    "template": "Este texto refleja {}.",
    "label_positiva": "dificultad actual de pago",
}

CONFIGURACION_FINAL_EN = {
    "labels": [
        "current payment difficulty",
        "absence of current payment difficulty",
    ],
    "template": "This text reflects {}.",
    "label_positiva": "current payment difficulty",
}

print("✅ Configuración V2 congelada para evaluación final.")

✅ Configuración V2 congelada para evaluación final.


## 12. Evaluación final de modelos

Una vez seleccionada y congelada la configuración V2 mediante el conjunto
independiente de calibración, se realiza la evaluación final sobre los
1,930 registros restantes.

A partir de esta etapa:

- no se modifican las etiquetas zero-shot;
- no se modifica la plantilla;
- todos los modelos utilizan el mismo conjunto de evaluación;
- se calculan las mismas métricas;
- se mide la eficiencia de inferencia bajo el mismo entorno de ejecución.

Los resultados obtenidos en esta sección corresponden a las métricas finales
utilizadas para la comparación y selección del modelo recomendado.

### 12.1 Evaluación final - SELECTRA español

Se evalúa `Recognai/zeroshot_selectra_medium` utilizando la configuración
V2 seleccionada durante la calibración.

La inferencia se realiza directamente sobre los textos originales en español,
sin traducción intermedia.

In [57]:
# Liberar modelos utilizados durante la calibración anterior

del clasificador_bart
del traductor_es_en

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("✅ Memoria GPU liberada.")

✅ Memoria GPU liberada.


In [58]:
MODELO_SELECTRA = "Recognai/zeroshot_selectra_medium"

clasificador_selectra = pipeline(
    task="zero-shot-classification",
    model=MODELO_SELECTRA,
    device=0 if torch.cuda.is_available() else -1
)

print("✅ SELECTRA cargado correctamente.")

Device set to use cuda:0


✅ SELECTRA cargado correctamente.


In [59]:
textos_finales = df_evaluacion_final["texto"].tolist()

print("Registros de evaluación final:", len(textos_finales))

Registros de evaluación final: 1930


In [60]:
_ = clasificador_selectra(
    textos_finales[0],
    candidate_labels=CONFIGURACION_FINAL_ES["labels"],
    hypothesis_template=CONFIGURACION_FINAL_ES["template"]
)

if torch.cuda.is_available():
    torch.cuda.synchronize()

print("✅ Warm-up completado.")

✅ Warm-up completado.


In [61]:
import time

if torch.cuda.is_available():
    torch.cuda.synchronize()

inicio = time.perf_counter()

resultados_selectra_final = clasificador_selectra(
    textos_finales,
    candidate_labels=CONFIGURACION_FINAL_ES["labels"],
    hypothesis_template=CONFIGURACION_FINAL_ES["template"],
    batch_size=16
)

if torch.cuda.is_available():
    torch.cuda.synchronize()

fin = time.perf_counter()

tiempo_selectra_final = fin - inicio

latencia_selectra_final = (
    tiempo_selectra_final / len(textos_finales)
) * 1000

throughput_selectra_final = (
    len(textos_finales) / tiempo_selectra_final
)

print(f"Tiempo total: {tiempo_selectra_final:.4f} s")
print(f"Latencia promedio: {latencia_selectra_final:.4f} ms/registro")
print(f"Throughput: {throughput_selectra_final:.2f} registros/segundo")

Tiempo total: 9.9060 s
Latencia promedio: 5.1327 ms/registro
Throughput: 194.83 registros/segundo


In [62]:
predicciones_selectra_final = [
    resultado["labels"][0]
    for resultado in resultados_selectra_final
]

scores_selectra_final = [
    resultado["scores"][0]
    for resultado in resultados_selectra_final
]

df_evaluacion_final["pred_selectra"] = [
    1
    if pred == CONFIGURACION_FINAL_ES["label_positiva"]
    else 0
    for pred in predicciones_selectra_final
]

df_evaluacion_final["score_selectra"] = scores_selectra_final

In [63]:
display(
    df_evaluacion_final[
        [
            "id",
            "texto",
            "etiqueta",
            "pred_selectra",
            "score_selectra"
        ]
    ].head(10)
)

,id,texto,etiqueta,pred_selectra,score_selectra
0,FP0001,Quería comentar que hice una transferencia par...,0,1,0.949711
1,FP0002,Necesito informar que quiero saber cuánto sald...,0,1,0.965072
2,FP0003,Quería comentar que no cuento con suficiente e...,1,1,0.948451
3,FP0004,Quería comentar que tuvimos gastos familiares ...,1,1,0.678561
4,FP0005,Necesito informar que por ahora no tengo sufic...,1,1,0.966116
5,FP0006,Actualmente me está costando cubrir la mensual...,1,1,0.990960
6,FP0007,Necesito informar que necesito revisar el sald...,0,1,0.991379
7,FP0008,Actualmente necesito posponer temporalmente el...,1,1,0.993078
8,FP0009,Por el momento quiero consultar el detalle de ...,0,1,0.775954
9,FP0010,Quería checar algo: por el momento quisiera ac...,0,0,0.903592


In [64]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

y_true_final = df_evaluacion_final["etiqueta"]
y_pred_selectra_final = df_evaluacion_final["pred_selectra"]

accuracy_selectra_final = accuracy_score(
    y_true_final,
    y_pred_selectra_final
)

precision_selectra_final = precision_score(
    y_true_final,
    y_pred_selectra_final
)

recall_selectra_final = recall_score(
    y_true_final,
    y_pred_selectra_final
)

f1_selectra_final = f1_score(
    y_true_final,
    y_pred_selectra_final
)

f1_macro_selectra_final = f1_score(
    y_true_final,
    y_pred_selectra_final,
    average="macro"
)

print("=== SELECTRA - EVALUACIÓN FINAL ===")
print(f"Accuracy       : {accuracy_selectra_final:.4f}")
print(f"Precision      : {precision_selectra_final:.4f}")
print(f"Recall         : {recall_selectra_final:.4f}")
print(f"F1 riesgo      : {f1_selectra_final:.4f}")
print(f"F1 macro       : {f1_macro_selectra_final:.4f}")

print("\nMatriz de confusión:")
print(
    confusion_matrix(
        y_true_final,
        y_pred_selectra_final
    )
)

print("\nReporte de clasificación:")
print(
    classification_report(
        y_true_final,
        y_pred_selectra_final,
        target_names=[
            "SIN_SEÑAL_RIESGO",
            "SEÑAL_RIESGO"
        ]
    )
)

=== SELECTRA - EVALUACIÓN FINAL ===
Accuracy       : 0.6073
Precision      : 0.5602
Recall         : 0.9979
F1 riesgo      : 0.7176
F1 macro       : 0.5365

Matriz de confusión:
[[209 756]
 [  2 963]]

Reporte de clasificación:
                  precision    recall  f1-score   support

SIN_SEÑAL_RIESGO       0.99      0.22      0.36       965
    SEÑAL_RIESGO       0.56      1.00      0.72       965

        accuracy                           0.61      1930
       macro avg       0.78      0.61      0.54      1930
    weighted avg       0.78      0.61      0.54      1930



In [65]:
resultados_finales = []

resultados_finales.append({
    "modelo": MODELO_SELECTRA,
    "estrategia": "Español",
    "accuracy": accuracy_selectra_final,
    "precision": precision_selectra_final,
    "recall": recall_selectra_final,
    "f1_riesgo": f1_selectra_final,
    "f1_macro": f1_macro_selectra_final,
    "tiempo_total_s": tiempo_selectra_final,
    "latencia_ms": latencia_selectra_final,
    "throughput_reg_s": throughput_selectra_final
})

display(pd.DataFrame(resultados_finales))

,modelo,estrategia,accuracy,precision,recall,f1_riesgo,f1_macro,tiempo_total_s,latencia_ms,throughput_reg_s
0,Recognai/zeroshot_selectra_medium,Español,0.607254,0.560209,0.997927,0.717586,0.536514,9.906021,5.132653,194.831005


### 12.2 Evaluación final - mDeBERTa multilingüe

Se evalúa `MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7`
sobre los mismos 1,930 registros utilizados con SELECTRA.

Se mantiene fija la configuración V2 seleccionada durante la calibración,
permitiendo una comparación directa entre ambas arquitecturas.

In [66]:
del clasificador_selectra

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("✅ Memoria de SELECTRA liberada.")

✅ Memoria de SELECTRA liberada.


In [67]:
MODELO_MDEBERTA = (
    "MoritzLaurer/"
    "mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
)

clasificador_mdeberta = pipeline(
    task="zero-shot-classification",
    model=MODELO_MDEBERTA,
    device=0 if torch.cuda.is_available() else -1
)

print("✅ mDeBERTa cargado correctamente.")

Device set to use cuda:0


✅ mDeBERTa cargado correctamente.


In [68]:
_ = clasificador_mdeberta(
    textos_finales[0],
    candidate_labels=CONFIGURACION_FINAL_ES["labels"],
    hypothesis_template=CONFIGURACION_FINAL_ES["template"]
)

if torch.cuda.is_available():
    torch.cuda.synchronize()

print("✅ Warm-up completado.")

✅ Warm-up completado.


In [69]:
if torch.cuda.is_available():
    torch.cuda.synchronize()

inicio = time.perf_counter()

resultados_mdeberta_final = clasificador_mdeberta(
    textos_finales,
    candidate_labels=CONFIGURACION_FINAL_ES["labels"],
    hypothesis_template=CONFIGURACION_FINAL_ES["template"],
    batch_size=16
)

if torch.cuda.is_available():
    torch.cuda.synchronize()

fin = time.perf_counter()

tiempo_mdeberta_final = fin - inicio

latencia_mdeberta_final = (
    tiempo_mdeberta_final / len(textos_finales)
) * 1000

throughput_mdeberta_final = (
    len(textos_finales) / tiempo_mdeberta_final
)

print(f"Tiempo total: {tiempo_mdeberta_final:.4f} s")
print(f"Latencia promedio: {latencia_mdeberta_final:.4f} ms/registro")
print(f"Throughput: {throughput_mdeberta_final:.2f} registros/segundo")

Tiempo total: 20.8839 s
Latencia promedio: 10.8207 ms/registro
Throughput: 92.42 registros/segundo


In [70]:
predicciones_mdeberta_final = [
    resultado["labels"][0]
    for resultado in resultados_mdeberta_final
]

scores_mdeberta_final = [
    resultado["scores"][0]
    for resultado in resultados_mdeberta_final
]

df_evaluacion_final["pred_mdeberta"] = [
    1
    if pred == CONFIGURACION_FINAL_ES["label_positiva"]
    else 0
    for pred in predicciones_mdeberta_final
]

df_evaluacion_final["score_mdeberta"] = scores_mdeberta_final

In [71]:
y_pred_mdeberta_final = df_evaluacion_final["pred_mdeberta"]

accuracy_mdeberta_final = accuracy_score(
    y_true_final,
    y_pred_mdeberta_final
)

precision_mdeberta_final = precision_score(
    y_true_final,
    y_pred_mdeberta_final
)

recall_mdeberta_final = recall_score(
    y_true_final,
    y_pred_mdeberta_final
)

f1_mdeberta_final = f1_score(
    y_true_final,
    y_pred_mdeberta_final
)

f1_macro_mdeberta_final = f1_score(
    y_true_final,
    y_pred_mdeberta_final,
    average="macro"
)

print("=== mDeBERTa - EVALUACIÓN FINAL ===")
print(f"Accuracy       : {accuracy_mdeberta_final:.4f}")
print(f"Precision      : {precision_mdeberta_final:.4f}")
print(f"Recall         : {recall_mdeberta_final:.4f}")
print(f"F1 riesgo      : {f1_mdeberta_final:.4f}")
print(f"F1 macro       : {f1_macro_mdeberta_final:.4f}")

print("\nMatriz de confusión:")
print(
    confusion_matrix(
        y_true_final,
        y_pred_mdeberta_final
    )
)

=== mDeBERTa - EVALUACIÓN FINAL ===
Accuracy       : 0.6642
Precision      : 0.5983
Recall         : 1.0000
F1 riesgo      : 0.7486
F1 macro       : 0.6216

Matriz de confusión:
[[317 648]
 [  0 965]]


In [72]:
resultados_finales.append({
    "modelo": MODELO_MDEBERTA,
    "estrategia": "Multilingüe",
    "accuracy": accuracy_mdeberta_final,
    "precision": precision_mdeberta_final,
    "recall": recall_mdeberta_final,
    "f1_riesgo": f1_mdeberta_final,
    "f1_macro": f1_macro_mdeberta_final,
    "tiempo_total_s": tiempo_mdeberta_final,
    "latencia_ms": latencia_mdeberta_final,
    "throughput_reg_s": throughput_mdeberta_final
})

display(pd.DataFrame(resultados_finales))

,modelo,estrategia,accuracy,precision,recall,f1_riesgo,f1_macro,tiempo_total_s,latencia_ms,throughput_reg_s
0,Recognai/zeroshot_selectra_medium,Español,0.607254,0.560209,0.997927,0.717586,0.536514,9.906021,5.132653,194.831005
1,MoritzLaurer/mDeBERTa-v3-base-xnli-multilingua...,Multilingüe,0.664249,0.598264,1.000000,0.748642,0.621591,20.883863,10.820654,92.415852


### 12.3 Evaluación final - BART con traducción español → inglés

Se evalúa la estrategia compuesta por:

1. traducción de español a inglés mediante `Helsinki-NLP/opus-mt-es-en`;
2. clasificación zero-shot mediante `facebook/bart-large-mnli`.

La configuración semántica utilizada corresponde a V2, seleccionada durante
la calibración común.

Para representar de manera realista el costo operativo de esta estrategia,
la medición de tiempo incluye tanto la traducción como la clasificación.

In [73]:
del clasificador_mdeberta

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("✅ Memoria de mDeBERTa liberada.")

✅ Memoria de mDeBERTa liberada.


In [74]:
MODELO_TRADUCCION = "Helsinki-NLP/opus-mt-es-en"
MODELO_BART = "facebook/bart-large-mnli"

traductor_es_en = pipeline(
    task="translation",
    model=MODELO_TRADUCCION,
    device=0 if torch.cuda.is_available() else -1
)

clasificador_bart = pipeline(
    task="zero-shot-classification",
    model=MODELO_BART,
    device=0 if torch.cuda.is_available() else -1
)

print("✅ Traductor cargado.")
print("✅ BART-MNLI cargado.")

Device set to use cuda:0
Device set to use cuda:0


✅ Traductor cargado.
✅ BART-MNLI cargado.


In [75]:
# Warm-up del traductor
traduccion_warmup = traductor_es_en(
    textos_finales[0],
    max_length=256
)[0]["translation_text"]

# Warm-up de BART
_ = clasificador_bart(
    traduccion_warmup,
    candidate_labels=CONFIGURACION_FINAL_EN["labels"],
    hypothesis_template=CONFIGURACION_FINAL_EN["template"]
)

if torch.cuda.is_available():
    torch.cuda.synchronize()

print("✅ Warm-up completo de traducción + BART.")

✅ Warm-up completo de traducción + BART.


In [76]:
if torch.cuda.is_available():
    torch.cuda.synchronize()

inicio = time.perf_counter()

# 1. Traducción español → inglés
traducciones_finales = traductor_es_en(
    textos_finales,
    batch_size=16,
    max_length=256
)

textos_finales_en = [
    resultado["translation_text"]
    for resultado in traducciones_finales
]

# 2. Clasificación en inglés
resultados_bart_final = clasificador_bart(
    textos_finales_en,
    candidate_labels=CONFIGURACION_FINAL_EN["labels"],
    hypothesis_template=CONFIGURACION_FINAL_EN["template"],
    batch_size=16
)

if torch.cuda.is_available():
    torch.cuda.synchronize()

fin = time.perf_counter()

tiempo_bart_final = fin - inicio

latencia_bart_final = (
    tiempo_bart_final / len(textos_finales)
) * 1000

throughput_bart_final = (
    len(textos_finales) / tiempo_bart_final
)

print(f"Tiempo total end-to-end: {tiempo_bart_final:.4f} s")
print(f"Latencia promedio: {latencia_bart_final:.4f} ms/registro")
print(f"Throughput: {throughput_bart_final:.2f} registros/segundo")

Tiempo total end-to-end: 118.5156 s
Latencia promedio: 61.4070 ms/registro
Throughput: 16.28 registros/segundo


In [77]:
predicciones_bart_final = [
    resultado["labels"][0]
    for resultado in resultados_bart_final
]

scores_bart_final = [
    resultado["scores"][0]
    for resultado in resultados_bart_final
]

df_evaluacion_final["texto_en"] = textos_finales_en

df_evaluacion_final["pred_bart"] = [
    1
    if pred == CONFIGURACION_FINAL_EN["label_positiva"]
    else 0
    for pred in predicciones_bart_final
]

df_evaluacion_final["score_bart"] = scores_bart_final

In [78]:
y_pred_bart_final = df_evaluacion_final["pred_bart"]

accuracy_bart_final = accuracy_score(
    y_true_final,
    y_pred_bart_final
)

precision_bart_final = precision_score(
    y_true_final,
    y_pred_bart_final
)

recall_bart_final = recall_score(
    y_true_final,
    y_pred_bart_final
)

f1_bart_final = f1_score(
    y_true_final,
    y_pred_bart_final
)

f1_macro_bart_final = f1_score(
    y_true_final,
    y_pred_bart_final,
    average="macro"
)

print("=== BART + TRADUCCIÓN - EVALUACIÓN FINAL ===")
print(f"Accuracy       : {accuracy_bart_final:.4f}")
print(f"Precision      : {precision_bart_final:.4f}")
print(f"Recall         : {recall_bart_final:.4f}")
print(f"F1 riesgo      : {f1_bart_final:.4f}")
print(f"F1 macro       : {f1_macro_bart_final:.4f}")

print("\nMatriz de confusión:")
print(
    confusion_matrix(
        y_true_final,
        y_pred_bart_final
    )
)

=== BART + TRADUCCIÓN - EVALUACIÓN FINAL ===
Accuracy       : 0.7425
Precision      : 0.6601
Recall         : 1.0000
F1 riesgo      : 0.7952
F1 macro       : 0.7242

Matriz de confusión:
[[468 497]
 [  0 965]]


In [79]:
resultados_finales.append({
    "modelo": "facebook/bart-large-mnli + Helsinki-NLP/opus-mt-es-en",
    "estrategia": "Inglés + traducción",
    "accuracy": accuracy_bart_final,
    "precision": precision_bart_final,
    "recall": recall_bart_final,
    "f1_riesgo": f1_bart_final,
    "f1_macro": f1_macro_bart_final,
    "tiempo_total_s": tiempo_bart_final,
    "latencia_ms": latencia_bart_final,
    "throughput_reg_s": throughput_bart_final
})

df_resultados_finales = pd.DataFrame(resultados_finales)

display(df_resultados_finales)

,modelo,estrategia,accuracy,precision,recall,f1_riesgo,f1_macro,tiempo_total_s,latencia_ms,throughput_reg_s
0,Recognai/zeroshot_selectra_medium,Español,0.607254,0.560209,0.997927,0.717586,0.536514,9.906021,5.132653,194.831005
1,MoritzLaurer/mDeBERTa-v3-base-xnli-multilingua...,Multilingüe,0.664249,0.598264,1.000000,0.748642,0.621591,20.883863,10.820654,92.415852
2,facebook/bart-large-mnli + Helsinki-NLP/opus-m...,Inglés + traducción,0.742487,0.660055,1.000000,0.795220,0.724198,118.515561,61.407027,16.284781


## 13. Análisis segmentado del desempeño

Además de las métricas globales, se analiza el comportamiento de los modelos
ante diferentes características lingüísticas del dataset.

El objetivo es identificar si el desempeño cambia cuando los textos presentan
mayor complejidad contextual o expresiones propias del español utilizado en
México.

### 13.1 Desempeño por dificultad lingüística

Los registros fueron clasificados previamente en tres niveles:

- baja;
- media;
- alta.

La dificultad se determinó mediante reglas lingüísticas relacionadas con
negación, temporalidad, incertidumbre, situaciones hipotéticas, casos
resueltos y lenguaje local.

In [81]:
def metricas_por_segmento(
    df,
    columna_segmento,
    columna_prediccion,
    nombre_modelo
):
    resultados = []

    for segmento, grupo in df.groupby(columna_segmento):

        y_true = grupo["etiqueta"]
        y_pred = grupo[columna_prediccion]

        resultados.append({
            "modelo": nombre_modelo,
            "segmento": segmento,
            "registros": len(grupo),
            "accuracy": accuracy_score(y_true, y_pred),
            "precision_riesgo": precision_score(
                y_true,
                y_pred,
                zero_division=0
            ),
            "recall_riesgo": recall_score(
                y_true,
                y_pred,
                zero_division=0
            ),
            "f1_riesgo": f1_score(
                y_true,
                y_pred,
                zero_division=0
            ),
            "f1_macro": f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            )
        })

    return pd.DataFrame(resultados)

In [82]:
dificultad_selectra = metricas_por_segmento(
    df_evaluacion_final,
    "dificultad",
    "pred_selectra",
    "SELECTRA"
)

dificultad_mdeberta = metricas_por_segmento(
    df_evaluacion_final,
    "dificultad",
    "pred_mdeberta",
    "mDeBERTa"
)

dificultad_bart = metricas_por_segmento(
    df_evaluacion_final,
    "dificultad",
    "pred_bart",
    "BART + traducción"
)

comparacion_dificultad = pd.concat(
    [
        dificultad_selectra,
        dificultad_mdeberta,
        dificultad_bart
    ],
    ignore_index=True
)

display(
    comparacion_dificultad.round(4)
)

,modelo,segmento,registros,accuracy,precision_riesgo,recall_riesgo,f1_riesgo,f1_macro
0,SELECTRA,alta,621,0.5475,0.3966,0.9946,0.5670,0.5466
1,SELECTRA,baja,471,0.4777,0.4371,1.0000,0.6083,0.4124
2,SELECTRA,media,838,0.7243,0.7188,0.9983,0.8358,0.4885
3,mDeBERTa,alta,621,0.6602,0.4672,1.0000,0.6368,0.6588
4,mDeBERTa,baja,471,0.5074,0.4515,1.0000,0.6221,0.4574
5,mDeBERTa,media,838,0.7554,0.7418,1.0000,0.8518,0.5761
6,BART + traducción,alta,621,0.7585,0.5522,1.0000,0.7115,0.7519
7,BART + traducción,baja,471,0.6263,0.5204,1.0000,0.6846,0.6131
8,BART + traducción,media,838,0.7959,0.7750,1.0000,0.8732,0.6752


### 13.2 Desempeño en lenguaje estándar y lenguaje local

Se analiza el comportamiento de los modelos ante textos con expresiones
o construcciones contextualizadas al español utilizado en México.

El dataset fue diseñado con una representación balanceada de lenguaje local
entre las dos clases para evitar que esta característica quedara asociada
artificialmente con una sola etiqueta.

In [83]:
df_evaluacion_final["tipo_lenguaje"] = (
    df_evaluacion_final["expresion_local"]
    .map({
        0: "estándar",
        1: "local"
    })
)

local_selectra = metricas_por_segmento(
    df_evaluacion_final,
    "tipo_lenguaje",
    "pred_selectra",
    "SELECTRA"
)

local_mdeberta = metricas_por_segmento(
    df_evaluacion_final,
    "tipo_lenguaje",
    "pred_mdeberta",
    "mDeBERTa"
)

local_bart = metricas_por_segmento(
    df_evaluacion_final,
    "tipo_lenguaje",
    "pred_bart",
    "BART + traducción"
)

comparacion_lenguaje = pd.concat(
    [
        local_selectra,
        local_mdeberta,
        local_bart
    ],
    ignore_index=True
)

display(
    comparacion_lenguaje.round(4)
)

,modelo,segmento,registros,accuracy,precision_riesgo,recall_riesgo,f1_riesgo,f1_macro
0,SELECTRA,estándar,1544,0.6082,0.5607,0.9987,0.7182,0.5376
1,SELECTRA,local,386,0.6036,0.5581,0.9948,0.7151,0.5320
2,mDeBERTa,estándar,1544,0.6658,0.5994,1.0000,0.7495,0.6238
3,mDeBERTa,local,386,0.6580,0.5938,1.0000,0.7452,0.6127
4,BART + traducción,estándar,1544,0.7383,0.6565,1.0000,0.7926,0.7191
5,BART + traducción,local,386,0.7591,0.6748,1.0000,0.8058,0.7442


#### Interpretación del lenguaje local

Los resultados no muestran un deterioro significativo del desempeño ante
lenguaje local.

SELECTRA presentó un F1 macro de 53.76 % en lenguaje estándar y 53.20 %
en lenguaje local, mientras que mDeBERTa pasó de 62.38 % a 61.27 %.

En ambos casos la diferencia fue reducida.

Contrario a la hipótesis inicial, BART con traducción no presentó una caída
de desempeño en los textos con lenguaje local. El modelo obtuvo un F1 macro
de 71.91 % en lenguaje estándar y 74.42 % en lenguaje local.

Asimismo, su accuracy aumentó de 73.83 % a 75.91 % en este segmento.

Estos resultados sugieren que, dentro del dataset sintético evaluado, la
etapa de traducción preservó adecuadamente el significado de las expresiones
locales incluidas en el experimento.

No obstante, este resultado debe interpretarse dentro del alcance del
prototipo y no implica que toda expresión regional o conversación real de
clientes mexicanos pueda traducirse sin pérdida semántica.

### 13.3 Desempeño por dificultad lingüística

Se compara el desempeño de los tres modelos según el nivel de dificultad
asignado previamente a los textos:

- baja;
- media;
- alta.

El objetivo es identificar si el rendimiento de los modelos se mantiene
cuando aumenta la complejidad contextual del mensaje.

In [84]:
dificultad_selectra = metricas_por_segmento(
    df_evaluacion_final,
    "dificultad",
    "pred_selectra",
    "SELECTRA"
)

dificultad_mdeberta = metricas_por_segmento(
    df_evaluacion_final,
    "dificultad",
    "pred_mdeberta",
    "mDeBERTa"
)

dificultad_bart = metricas_por_segmento(
    df_evaluacion_final,
    "dificultad",
    "pred_bart",
    "BART + traducción"
)

comparacion_dificultad = pd.concat(
    [
        dificultad_selectra,
        dificultad_mdeberta,
        dificultad_bart
    ],
    ignore_index=True
)

display(
    comparacion_dificultad.round(4)
)

,modelo,segmento,registros,accuracy,precision_riesgo,recall_riesgo,f1_riesgo,f1_macro
0,SELECTRA,alta,621,0.5475,0.3966,0.9946,0.5670,0.5466
1,SELECTRA,baja,471,0.4777,0.4371,1.0000,0.6083,0.4124
2,SELECTRA,media,838,0.7243,0.7188,0.9983,0.8358,0.4885
3,mDeBERTa,alta,621,0.6602,0.4672,1.0000,0.6368,0.6588
4,mDeBERTa,baja,471,0.5074,0.4515,1.0000,0.6221,0.4574
5,mDeBERTa,media,838,0.7554,0.7418,1.0000,0.8518,0.5761
6,BART + traducción,alta,621,0.7585,0.5522,1.0000,0.7115,0.7519
7,BART + traducción,baja,471,0.6263,0.5204,1.0000,0.6846,0.6131
8,BART + traducción,media,838,0.7959,0.7750,1.0000,0.8732,0.6752


#### Interpretación por dificultad lingüística

BART con traducción presentó el mejor F1 macro en los tres niveles de
dificultad lingüística definidos para el dataset.

Resultados de F1 macro:

| Dificultad | SELECTRA | mDeBERTa | BART + traducción |
|---|---:|---:|---:|
| Baja | 0.4124 | 0.4574 | 0.6131 |
| Media | 0.4885 | 0.5761 | 0.6752 |
| Alta | 0.5466 | 0.6588 | 0.7519 |

En los registros clasificados como dificultad alta, BART alcanzó un accuracy
de 75.85 % y un F1 macro de 75.19 %, superando a mDeBERTa y SELECTRA.

La ventaja de BART también se mantuvo en los segmentos de dificultad media
y baja.

Estos resultados indican que la estrategia basada en traducción no solamente
presenta el mejor desempeño global, sino que mantiene su ventaja al segmentar
el conjunto de evaluación según características lingüísticas.

Sin embargo, los niveles de dificultad utilizados en este prototipo fueron
definidos mediante reglas lingüísticas y no representan una medida empírica
perfecta de dificultad para cada arquitectura. Por este motivo, el análisis
segmentado se interpreta como evidencia complementaria y no como una escala
absoluta de complejidad.

## 14. Recomendación final del modelo

### Modelo recomendado: BART + traducción

Con base en los resultados experimentales, se recomienda la estrategia
compuesta por:

`Helsinki-NLP/opus-mt-es-en`
+
`facebook/bart-large-mnli`

para el prototipo de FinPredict NLP.

La recomendación no se basa exclusivamente en accuracy, sino en la evaluación
conjunta de desempeño global, capacidad para detectar señales de riesgo,
robustez lingüística y eficiencia computacional.

### Evidencia de desempeño

BART + traducción obtuvo:

- Accuracy: 74.25 %
- Precision de riesgo: 66.01 %
- Recall de riesgo: 100 %
- F1-score de riesgo: 79.52 %
- F1 macro: 72.42 %
- Latencia promedio experimental: 61.41 ms por registro
- Throughput experimental: 16.28 registros por segundo

Además, presentó el mejor F1 macro en:

- lenguaje estándar;
- lenguaje local;
- dificultad baja;
- dificultad media;
- dificultad alta.

### Trade-off

La principal desventaja de esta estrategia es su mayor costo computacional.

Su latencia promedio experimental fue aproximadamente:

- 12 veces mayor que SELECTRA;
- 5.7 veces mayor que mDeBERTa.

Esto se debe principalmente a que la estrategia requiere dos etapas:

`Texto en español → Traducción al inglés → Clasificación`

Por lo tanto, BART + traducción se recomienda para el prototipo cuando la
calidad de clasificación tenga mayor prioridad que la mínima latencia.

Si el contexto de implementación exigiera una reducción considerable del
tiempo de procesamiento o de los recursos computacionales, mDeBERTa sería
la segunda alternativa recomendada debido a su mejor equilibrio entre
desempeño y eficiencia.

SELECTRA ofrece la mayor eficiencia computacional, pero su elevada cantidad
de falsos positivos limita su conveniencia para un escenario donde cada
alerta genere una acción operativa.

### Umbral y condición de continuidad

El prototipo demuestra viabilidad técnica, pero los resultados zero-shot no
se consideran suficientes para un despliegue productivo automático.

Aunque el modelo recomendado alcanza un recall del 100 % para la clase de
riesgo, su precision de 66.01 % implica que aún existe una proporción
relevante de falsos positivos.

Por lo tanto, el modelo debe utilizarse inicialmente como herramienta de
priorización y apoyo a la revisión humana, no como mecanismo automático para
tomar decisiones sobre clientes.

Como siguiente etapa, se recomienda evaluar fine-tuning con datos
representativos del dominio hipotecario y establecer umbrales de desempeño
validados junto con las áreas de Riesgo y Operaciones antes de considerar
un despliegue productivo.

In [85]:
from pathlib import Path

RESULTS_PATH = Path("results")
RESULTS_PATH.mkdir(exist_ok=True)

df_resultados_finales.to_csv(
    RESULTS_PATH / "metricas_modelos.csv",
    index=False,
    encoding="utf-8-sig"
)

comparacion_dificultad.to_csv(
    RESULTS_PATH / "metricas_por_dificultad.csv",
    index=False,
    encoding="utf-8-sig"
)

comparacion_lenguaje.to_csv(
    RESULTS_PATH / "metricas_lenguaje_local.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✅ Resultados exportados correctamente.")

✅ Resultados exportados correctamente.


In [86]:
!ls -lh results

total 12K
-rw-r--r-- 1 root root  747 Oct 11 21:55 metricas_lenguaje_local.csv
-rw-r--r-- 1 root root  721 Oct 11 21:55 metricas_modelos.csv
-rw-r--r-- 1 root root 1021 Oct 11 21:55 metricas_por_dificultad.csv
